# Phase 3 &mdash; Bronze Layer

This is the first stage of our Medallion pipeline. Its job is to fetch the Lending Club source data from Kaggle, keep only loans issued from 2014 through 2017, and register that raw slice as a managed Delta table. No cleaning, no feature engineering, and no model-oriented filtering happen here &mdash; those all belong later in the pipeline.

### Input
The Lending Club 2007&ndash;2020Q1 Kaggle dataset: <https://www.kaggle.com/datasets/ethon0426/lending-club-20072020q1>. The notebook downloads it with `kagglehub`, reads the large source file in pandas chunks, and saves the reusable 2014&ndash;2017 CSV to `/Volumes/workspace/default/raw_data/optimized_data_14_17.csv`.

### Output
`workspace.default.bronze_loans` &mdash; an identical Delta copy of the auto-generated 2014&ndash;2017 CSV, renamed under our medallion convention.

### Why a sanitization step?
CSV source files can contain column names that Delta does not accept, such as spaces, colons, or punctuation. We run a one-pass rename that replaces any non-alphanumeric character with an underscore and lowercases every name. It is a *cosmetic* schema change only &mdash; the row values are unchanged &mdash; but it lets Delta accept the table.

In [0]:
# A SparkSession named `spark` is already available on a Databricks cluster.
# The try/except keeps the notebook runnable outside Databricks too (e.g. in a
# local `pip install pyspark delta-spark` environment for development).
try:
    spark  # type: ignore[name-defined]
except NameError:
    from pyspark.sql import SparkSession
    spark = (
        SparkSession.builder.appName("phase3-bronze")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate()
    )

print(f"Spark {spark.version} ready.")

Spark 4.1.0 ready.


In [0]:
# Kaggle source, volume output, and destination table.
# The Kaggle API needs credentials on the Databricks cluster, usually via
# KAGGLE_USERNAME and KAGGLE_KEY environment variables or a kaggle.json file.
KAGGLE_DATASET = "ethon0426/lending-club-20072020q1"
RAW_DATA_DIR = "/Volumes/workspace/default/raw_data"
SOURCE_PATH = f"{RAW_DATA_DIR}/optimized_data_14_17.csv"
BRONZE_TABLE = "workspace.default.bronze_loans"

LOCAL_WORK_DIR = "/local_disk0/tmp/lending_club_kaggle"
LOCAL_EXTRACT_DIR = f"{LOCAL_WORK_DIR}/extracted"
LOCAL_FILTERED_PATH = f"{LOCAL_WORK_DIR}/optimized_data_14_17.csv"
FILTERED_TEMP_PATH = f"{RAW_DATA_DIR}/_optimized_data_14_17_in_progress.csv"
LEGACY_TEMP_PATHS = [
    f"{RAW_DATA_DIR}/_kaggle_source_data",
    f"{RAW_DATA_DIR}/_optimized_data_14_17_tmp",
]

## Step 1 &mdash; Download and filter the Kaggle dataset

Download the raw Kaggle dataset onto the driver, read the source CSV in pandas chunks with `compression=None`, keep only loans issued from 2014 through 2017, and write the final reusable CSV to the Unity Catalog volume. This avoids loading the full file into memory and avoids Serverless Spark restrictions around driver-local and gzip source files.

In [0]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import pandas as pd

try:
    import kagglehub
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "kagglehub"])
    import kagglehub

os.environ.setdefault("KAGGLEHUB_CACHE", LOCAL_WORK_DIR)


def rm_path(path):
    try:
        dbutils.fs.rm(path, True)  # type: ignore[name-defined]
    except Exception:
        local_path = path.replace("file:", "")
        if os.path.isdir(local_path):
            shutil.rmtree(local_path, ignore_errors=True)
        elif os.path.exists(local_path):
            os.remove(local_path)


def mkdirs(path):
    try:
        dbutils.fs.mkdirs(path)  # type: ignore[name-defined]
    except Exception:
        Path(path.replace("file:", "")).mkdir(parents=True, exist_ok=True)


def copy_local_file_to_volume(src, dst):
    rm_path(dst)
    with open(src, mode="rb") as src_file, open(dst, mode="wb") as dst_file:
        shutil.copyfileobj(src_file, dst_file, length=1024 * 1024 * 16)


def find_source_file(download_path):
    path = Path(download_path)
    candidates = []
    search_roots = [path] if path.is_dir() else [path.parent]

    for root in search_roots:
        for pattern in ("*.csv", "*.gzip", "*.gz", "*.zip"):
            candidates.extend(root.rglob(pattern))

    data_files = [p for p in candidates if p.suffix.lower() in {".csv", ".gzip", ".gz"}]
    if data_files:
        return str(max(data_files, key=lambda p: p.stat().st_size))

    zip_files = [p for p in candidates if p.suffix.lower() == ".zip"]
    if not zip_files:
        raise FileNotFoundError(f"No CSV, gzip, gz, or zip file found under {download_path}")

    rm_path(LOCAL_EXTRACT_DIR)
    Path(LOCAL_EXTRACT_DIR).mkdir(parents=True, exist_ok=True)
    zip_path = max(zip_files, key=lambda p: p.stat().st_size)
    with zipfile.ZipFile(zip_path) as zf:
        members = [m for m in zf.namelist() if m.lower().endswith((".csv", ".gzip", ".gz"))]
        if not members:
            raise FileNotFoundError(f"No CSV/gzip data file found inside {zip_path}")
        member = max(members, key=lambda m: zf.getinfo(m).file_size)
        zf.extract(member, LOCAL_EXTRACT_DIR)
    return str(Path(LOCAL_EXTRACT_DIR) / member)


def write_filtered_csv(source_file, output_path):
    start_date = pd.Timestamp("2014-01-01")
    end_date = pd.Timestamp("2017-12-31")
    chunk_size = 100_000
    first_chunk = True
    rows_kept = 0

    # Kaggle names this file .gzip, but the current content is plain CSV.
    # Match the local script by forcing compression=None and filtering in chunks.
    for chunk in pd.read_csv(source_file, chunksize=chunk_size, compression=None, low_memory=False):
        if "issue_d" not in chunk.columns:
            raise ValueError("Kaggle source CSV does not contain an issue_d column.")

        chunk["issue_d"] = pd.to_datetime(chunk["issue_d"], format="%b-%Y", errors="coerce")
        chunk = chunk[(chunk["issue_d"] >= start_date) & (chunk["issue_d"] <= end_date)]
        if chunk.empty:
            continue

        chunk.to_csv(output_path, mode="w" if first_chunk else "a", header=first_chunk, index=False)
        first_chunk = False
        rows_kept += len(chunk)

    return rows_kept


rm_path(LOCAL_WORK_DIR)
Path(LOCAL_WORK_DIR).mkdir(parents=True, exist_ok=True)
mkdirs(RAW_DATA_DIR)
rm_path(FILTERED_TEMP_PATH)
for temp_path in LEGACY_TEMP_PATHS:
    rm_path(temp_path)

download_path = kagglehub.dataset_download(KAGGLE_DATASET)
source_file = find_source_file(download_path)
print(f"Downloaded Kaggle dataset to: {download_path}")
print(f"Streaming and filtering source file: {source_file}")

row_count = write_filtered_csv(source_file, LOCAL_FILTERED_PATH)
if row_count == 0:
    raise ValueError("No rows found for issue_d between 2014-01-01 and 2017-12-31. Check the Kaggle source schema/date format.")
copy_local_file_to_volume(LOCAL_FILTERED_PATH, FILTERED_TEMP_PATH)
rm_path(LOCAL_FILTERED_PATH)
try:
    dbutils.fs.mv(FILTERED_TEMP_PATH, SOURCE_PATH)  # type: ignore[name-defined]
except Exception:
    copy_local_file_to_volume(FILTERED_TEMP_PATH, SOURCE_PATH)
    rm_path(FILTERED_TEMP_PATH)

# Drop the full Kaggle/extracted data once the filtered CSV exists in the volume.
rm_path(LOCAL_EXTRACT_DIR)
if os.path.exists(download_path):
    shutil.rmtree(download_path, ignore_errors=True) if os.path.isdir(download_path) else os.remove(download_path)

raw_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(SOURCE_PATH)
)

print(f"Saved {row_count:,} rows to {SOURCE_PATH}")
print(f"Rows: {row_count:,}")
print(f"Columns: {len(raw_df.columns)}")
display(raw_df.limit(5))


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


100%|██████████| 482M/482M [00:03<00:00, 143MB/s]

Extracting files...


Downloaded Kaggle dataset to: /local_disk0/tmp/lending_club_kaggle/datasets/ethon0426/lending-club-20072020q1/versions/3
Streaming and filtering source file: /local_disk0/tmp/lending_club_kaggle/datasets/ethon0426/lending-club-20072020q1/versions/3/Loan_status_2007-2020Q3.gzip
Saved 1,534,710 rows to /Volumes/workspace/default/raw_data/optimized_data_14_17.csv
Rows: 1,534,710
Columns: 142


Unnamed: 0,id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,debt_settlement_flag
0,120122535,12000,12000,12000.0,36 months,7.97%,375.88,A,A5,associate,10+ years,OWN,42000.0,Source Verified,2017-09-01,Fully Paid,n,https://lendingclub.com/browse/loanDetail.action?loan_id=120122535,debt_consolidation,Debt consolidation,923xx,CA,27.74,0.0,Jun-1996,715,719,0.0,null,80.0,9.0,1,11457,37%,16,w,0.0,0.0,13500.44,13500.44,12000.0,1500.44,0.0,0.0,0.0,May-2020,2591.95,null,May-2020,694,690,0,null,1,Individual,null,null,null,0,0.0,30502.0,1.0,2.0,1.0,3.0,8.0,19045.0,73.0,2.0,4.0,7117.0,53.0,31000.0,1.0,1.0,2.0,7.0,3389.0,7144.0,53.9,0.0,0,131.0,255.0,1.0,1.0,0.0,14.0,null,8.0,null,0.0,2.0,6.0,2.0,2.0,7.0,7.0,9.0,6.0,9.0,0.0,0.0,0.0,3.0,100.0,0.0,1.0,0,57180.0,30502.0,15500.0,26180.0,null,null,null,null,null,null,null,null,null,null,null,null,N,null,null,null,null,null,null,null,null,null,null,null,null,null,null,N
1,119374887,32000,32000,32000.0,36 months,11.99%,1062.71,B,B5,Nurse,10+ years,MORTGAGE,155000.0,Source Verified,2017-09-01,Current,n,https://lendingclub.com/browse/loanDetail.action?loan_id=119374887,credit_card,Credit card refinancing,080xx,NJ,12.35,2.0,Sep-2005,715,719,1.0,10.0,null,20.0,0,48309,34.1%,42,w,6158.89,6158.89,31838.67,31838.67,25841.11,5997.56,0.0,0.0,0.0,May-2020,0.0,Jul-2020,May-2020,674,670,0,null,1,Individual,null,null,null,0,0.0,405751.0,2.0,1.0,1.0,1.0,8.0,15582.0,78.0,4.0,7.0,14049.0,40.0,142600.0,0.0,2.0,2.0,8.0,22542.0,81313.0,34.5,0.0,0,91.0,144.0,1.0,1.0,3.0,1.0,24.0,1.0,10.0,0.0,7.0,10.0,12.0,27.0,3.0,18.0,36.0,10.0,20.0,0.0,0.0,0.0,5.0,94.9,0.0,0.0,0,527034.0,63891.0,124200.0,20034.0,null,null,null,null,null,null,null,null,null,null,null,null,Y,CVD19SKIP,INCOMECURT,ACTIVE,2.0,0.0,Apr-2020,Jun-2020,Apr-2020,2.0,0.0,ACTIVE,123.08,6189.66,1062.71,N
2,119321612,40000,40000,40000.0,60 

## Step 2 &mdash; Sanity-check with a few rows and the schema

In [0]:
raw_df.limit(5).toPandas()

,Unnamed: 0,id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,...,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,debt_settlement_flag
0,0,120122535,12000,12000,12000.0,36 months,7.97%,375.88,A,A5,associate,10+ years,OWN,42000.0,Source Verified,2017-09-01,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,debt_consolidation,Debt consolidation,923xx,CA,27.74,0.0,Jun-1996,715,719,0.0,NaN,80.0,9.0,1,11457,37%,16,w,0.0,0.00,13500.44,...,0.0,0.0,0.0,3.0,100.0,0.0,1.0,0,57180.0,30502.0,15500.0,26180.0,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,None,None,None,NaN,NaN,None,None,None,NaN,NaN,None,NaN,NaN,NaN,N
1,1,119374887,32000,32000,32000.0,36 months,11.99%,1062.71,B,B5,Nurse,10+ years,MORTGAGE,155000.0,Source Verified,2017-09-01,Current,n,https://lendingclub.com/browse/loanDetail.acti...,credit_card,Credit card refinancing,080xx,NJ,12.35,2.0,Sep-2005,715,719,1.0,10.0,NaN,20.0,0,48309,34.1%,42,w,6158.89,6158.89,31838.67,...,0.0,0.0,0.0,5.0,94.9,0.0,0.0,0,527034.0,63891.0,124200.0,20034.0,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Y,CVD19SKIP,INCOMECURT,ACTIVE,2.0,0.00,Apr-2020,Jun-2020,Apr-2020,2.0,0.0,ACTIVE,123.08000,6189.66,1062.71,N
2,2,119321612,40000,40000,40000.0,60 months,15.05%,952.65,C,C4,Driver,9 years,MORTGAGE,120000.0,Verified,2017-09-01,Current,n,https://lendingclub.com/browse/loanDetail.acti...,debt_consolidation,Debt consolidation,778xx,TX,31.11,0.0,Apr-2002,765,769,0.0,NaN,NaN,12.0,0,13389,20.7%,26,w,22376.9,22376.90,30417.91,...,0.0,0.0,0.0,2.0,100.0,0.0,0.0,0,367745.0,154261.0,64600.0,168145.0,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,None,None,None,NaN,NaN,None,None,None,NaN,NaN,None,NaN,NaN,NaN,N
3,3,120122034,16000,16000,16000.0,36 months,7.97%,501.17,A,A5,Senior Investigator,5 years,RENT,79077.0,Not Verified,2017-09-01,Current,n,https://lendingclub.com/browse/loanDetail.acti...,debt_consolidation,Debt consolidation,223xx,VA,15.94,0.0,Jun-2000,700,704,0.0,38.0,NaN,12.0,0,16217,57.7%,20,w,2455.43,2455.43,15527.57,...,0.0,0.0,0.0,0.0,78.9,100.0,0.0,0,125018.0,128572.0,3700.0,96918.0,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,None,None,None,NaN,NaN,None,None,None,NaN,NaN,None,NaN,NaN,NaN,N
4,4,118659541,33000,33000,33000.0,36 months,7.21%,1022.12,A,A3,Registered Nurse,< 1 year,MORTGAGE,107000.0,Verified,2017-09-01,Late (31-120 days),n,https://lendingclub.com/browse/loanDetail.acti...,debt_consolidation,Debt consolidation,750xx,TX,19.06,0.0,Dec-2005,785,789,0.0,NaN,NaN,25.0,0,18533,16.1%,52,w,9873.08,9873.08,26616.93,...,0.0,0.0,0.0,3.0,100.0,0.0,0.0,0,404510.0,48219.0,81700.0,73015.0,48968.0,705.0,709.0,Feb-2007,0.0,2.0,13.0,81.4,2.0,28.0,0.0,0.0,Y,ST0650PV01,UNEMPLOYED,ACTIVE,3.0,1022.12,Feb-2020,Apr-2020,Feb-2020,3.0,21.0,DELINQUENT,177.96231,10197.78,59.68,N


In [0]:
raw_df.printSchema()

root
 |-- Unnamed: 0: integer (nullable = true)
 |-- id: integer (nullable = true)
 |-- loan_amnt: integer (nullable = true)
 |-- funded_amnt: integer (nullable = true)
 |-- funded_amnt_inv: double (nullable = true)
 |-- term: string (nullable = true)
 |-- int_rate: string (nullable = true)
 |-- installment: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: string (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- issue_d: string (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- pymnt_plan: string (nullable = true)
 |-- url: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- title: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- addr_state: string (nullable = true)
 |-- dti: string (nullable = true)
 |-- delinq_2

## Step 3 &mdash; Sanitize column names

Replace any character outside `[A-Za-z0-9_]` with `_` and lowercase the result. This is a cosmetic schema change only, but it safely handles Kaggle CSV headers that Delta may reject.

In [0]:
import re

def sanitize(name):
    cleaned = re.sub(r"[^A-Za-z0-9_]+", "_", name).strip("_").lower()
    return cleaned or "col"

renamed_df = raw_df
changed = []
for old in raw_df.columns:
    new = sanitize(old)
    if new != old:
        renamed_df = renamed_df.withColumnRenamed(old, new)
        changed.append((old, new))

print(f"Renamed {len(changed)} column(s):")
for old, new in changed:
    print(f"  {old!r:<20} -> {new!r}")

Renamed 1 column(s):
  'Unnamed: 0'         -> 'unnamed_0'


## Step 4 &mdash; Write the Bronze Delta table

`mode("overwrite")` makes re-runs idempotent &mdash; the table ends in the same state whether this is the first run or the tenth.

In [0]:
(
    renamed_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

print(f"Wrote Delta table: {BRONZE_TABLE}")

Wrote Delta table: workspace.default.bronze_loans


## Step 5 &mdash; Verify

In [0]:
spark.sql(f"SELECT COUNT(*) AS row_count FROM {BRONZE_TABLE}").show()
spark.sql(f"DESCRIBE DETAIL {BRONZE_TABLE}") \
    .select("name", "location", "numFiles", "sizeInBytes") \
    .show(truncate=False)

+---------+
|row_count|
+---------+
|  1534710|
+---------+

+------------------------------+--------+--------+-----------+
|name                          |location|numFiles|sizeInBytes|
+------------------------------+--------+--------+-----------+
|workspace.default.bronze_loans|        |8       |181822312  |
+------------------------------+--------+--------+-----------+

